# Langkah 2 & 3: Pelatihan CNN untuk Klasifikasi Penyebab Deforestasi
Notebook ini menyusun image stack (GeoTIFF) menjadi array NumPy dan melatih model Convolutional Neural Network (CNN).

In [ ]:
import numpy as np
import rasterio
import glob, os
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
LABELS = {0:'tidak_yakin', 1:'pertanian_kecil', 2:'pertanian_besar', 
          3:'tambang', 4:'logging', 5:'kebakaran', 6:'gangguan_alam'}

# Fungsi pemuatan dan normalisasi patch GeoTIFF
def load_patch(path):
    with rasterio.open(path) as src:
        arr = src.read().astype('float32')  # shape (6, H, W)
    for i in range(arr.shape[0]):
        band = arr[i]
        arr[i] = (band - band.mean()) / (band.std() + 1e-6)
    return arr

In [ ]:
# (KODE SIMULASI UNTUK BUKTI KONSEP JIKA FILE TIFF TIDAK ADA LENGKAP)
# Men-generate array dummy berukuran 6-band (mirip GeoTIFF) untuk pengujian lokal/Colab cepat
np.random.seed(42)
X_dummy = np.random.rand(100, 6, 64, 64)  # 100 citra, 6 band, resolusi 64x64
y_dummy = np.random.randint(1, 7, 100)    # Label 1 sampai 6

# Transpose dari (N, C, H, W) ke format Tensorflow/Keras (N, H, W, C)
X = np.transpose(X_dummy, (0, 2, 3, 1))
y = y_dummy

print('Bentuk Dataset X:', X.shape, '| bentuk y:', y.shape)

In [ ]:
# Pembagian Data Latih dan Uji
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
n_cls = 7 # Total ada 7 kelas (0 s.d 6)

In [ ]:
# Arsitektur CNN
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=X.shape[1:]),
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(n_cls, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
# Penanganan Imbalanced Data (Membobotkan kelas yang jarang)
cw = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
class_weight = {i: w for i, w in zip(np.unique(y_tr), cw)}

# Proses Training
history = model.fit(X_tr, y_tr, validation_split=0.2, epochs=5, batch_size=16, class_weight=class_weight)
model.save('model_cnn_deforestasi.h5')

In [ ]:
# Evaluasi Macro F1 Score dan Confusion Matrix
y_pred = np.argmax(model.predict(X_te), axis=1)

print(classification_report(y_te, y_pred, labels=np.unique(y_te)))

cm = confusion_matrix(y_te, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix Deforestation Drivers')
plt.show()